# Mistral AI on Amazon Bedrock — text models and the size ladder

> **Sample code — not for production.** Provided as AWS Content under the AWS
> Customer Agreement; do not use it in production accounts or on production or other
> critical data. Running these cells calls Amazon Bedrock and incurs charges. Full
> disclaimer in the [README](../README.md#disclaimer).

Mistral has the widest size range of any family on `bedrock-mantle`: from a 3B
Ministral up to a 675B Mistral Large 3. That size spread makes it a good family
for demonstrating **cost-aware model routing** — send each request to the
cheapest model that can actually do the job.

**Models covered in this notebook**

| Model ID | Notes |
|---|---|
| `mistral.mistral-large-3-675b-instruct` | Flagship, 675B |
| `mistral.ministral-3-14b-instruct` | Mid-size |
| `mistral.ministral-3-8b-instruct` | Small |
| `mistral.ministral-3-3b-instruct` | Smallest, lowest latency |
| `mistral.magistral-small-2509` | Reasoning-oriented variant |

`02-devstral-and-voxtral.ipynb` covers the coding model (Devstral 2) and the
audio-capable Voxtral variants.

## Which API? Chat Completions.
This family is served by the **OpenAI-compatible Chat Completions API** at the
bare `/v1` path. The Responses API returns **400** for these models — proven in §2.

## Self-contained, but see also
- **Auth (SigV4 (AWS Signature Version 4) + short-term keys), the three URL paths,
  model discovery** →
  `../00-foundations/01-endpoints-auth-and-the-three-paths.ipynb`
- **Projects, cost attribution, data retention / ZDR (zero data retention),
  CloudWatch** →
  `../00-foundations/02-governance-projects-and-retention.ipynb`
- **Quotas, retry/backoff, service tiers, TTFT (time-to-first-token)** →
  `../00-foundations/03-scaling-tiers-and-latency.ipynb`

## Prerequisites
```bash
pip install -r ../requirements.txt
```

Needs openai, aws-bedrock-token-generator.

`requirements.txt` pins the exact versions this collection was tested
against. An unpinned install resolves whatever is current, which may be
untested or compromised (OWASP LLM03, Supply Chain).

### Where the helpers come from

The next cell does this:

```python
sys.path.insert(0, "../_shared")
from bedrock import ...
```

`bedrock` is **not** a package from PyPI — it is this collection's own helper
module, [`_shared/bedrock.py`](../_shared/bedrock.py). Every notebook sits one
level down, so `../_shared` puts it on the import path. It exists only to remove
repetition; the notebooks are the teaching material, and nothing in the module is
required to call Bedrock yourself.

What this notebook uses from it:

| Helper | What it does |
|---|---|
| `err` | pulls the human-readable message out of an error body, redacted |
| `list_models` | the `bedrock-mantle` model inventory for a Region |
| `parse_json_lenient` | parses the first complete JSON object out of model output, repairing truncated braces |
| `post` | signed JSON HTTP with retries. **Never raises on 4xx/5xx** — it returns `(status, body)` so a cell can *show* an error instead of stopping the notebook |
| `safe_print` | `print()` with account IDs, IAM principals and opaque service IDs redacted |
| `ttft` | times a streaming call: time-to-first-token and output frames/sec |
| `converse` | one Converse call; returns `(text, response)` and **never raises** on a service error |
| `converse_reasoning` | the reasoning trace from a Converse response, or `""` |
| `endpoints_for` | answers "mantle, runtime, or both" for a model, from the live catalogues |
| `resolve_runtime_id` | turns a model ID into the form Converse will accept, adding the `us.` profile prefix when one is required |
| `converse_text` | concatenates the text blocks of a Converse response — safer than `content[0]` |
| `converse_tool_uses` | the `toolUse` blocks from a Converse response |
| `runtime_client` | a boto3 `bedrock-runtime` client (Converse, InvokeModel) |

Two behaviours worth knowing before you read any output below, because several
cells depend on them:

- **`post()` and `converse()` never raise on a service error.** They return the
  status and body so a cell can *show* a 400 rather than stopping the notebook.
  Many cells here deliberately provoke an error to demonstrate a limit.
- **Anything printed from a control-plane response goes through redaction**, since
  this output is committed to a public repository.


In [1]:
import json
import sys
import time

sys.path.insert(0, "../_shared")
from bedrock import err, list_models, parse_json_lenient, post, safe_print, ttft

REGION = "us-east-1"

LARGE3 = "mistral.mistral-large-3-675b-instruct"
M14B = "mistral.ministral-3-14b-instruct"
M8B = "mistral.ministral-3-8b-instruct"
M3B = "mistral.ministral-3-3b-instruct"
MAGISTRAL = "mistral.magistral-small-2509"

LADDER = [M3B, M8B, M14B, LARGE3]

# Chat-Completions families live at the BARE /v1 path — not /openai/v1
# (that prefix is only for gemma-4, gpt-5.x and grok). See ../00-foundations/01.
PREFIX = "/v1"
BASE_URL = f"https://bedrock-mantle.{REGION}.api.aws{PREFIX}"
print("base URL:", BASE_URL)
print("size ladder:", LADDER)

base URL: https://bedrock-mantle.us-east-1.api.aws/v1
size ladder: ['mistral.ministral-3-3b-instruct', 'mistral.ministral-3-8b-instruct', 'mistral.ministral-3-14b-instruct', 'mistral.mistral-large-3-675b-instruct']


### Which endpoint, and the model ID for each

AWS recommends `bedrock-runtime` for new applications, and since August 2026 it
serves the OpenAI- and Anthropic-compatible APIs as well as Converse. So before
the first call, the question is which endpoint you want — and that has a
complication worth knowing about:

**the same model often carries a different ID on each endpoint.** Send a
`bedrock-mantle` ID to `bedrock-runtime` and you get *"The provided model
identifier is invalid"*, which reads like a missing model rather than a missing
translation.

The cell below asks both catalogues rather than stating an answer that will age.
`runtime_id_for()` returns `None` when a model is genuinely not on
`bedrock-runtime`, which is the honest signal for "you need mantle for this one".

In [2]:
from bedrock import endpoints_for, runtime_id_for

COVERED = [
    "mistral.magistral-small-2509",
    "mistral.ministral-3-14b-instruct",
    "mistral.ministral-3-3b-instruct",
    "mistral.ministral-3-8b-instruct",
    "mistral.mistral-large-3-675b-instruct",
]

print(f"{'model (as named on mantle)':38} {'on runtime as':40} endpoints")
print("-" * 96)
mantle_only = []
for model_id in COVERED:
    runtime_id = runtime_id_for(model_id, REGION)
    where = endpoints_for(model_id, REGION)
    label = ", ".join(name for name, present in where.items() if present) or "neither"
    if runtime_id is None:
        mantle_only.append(model_id)
    print(f"{model_id:38} {(runtime_id or '-- not on runtime --'):40} {label}")

renamed = [
    m for m in COVERED
    if (r := runtime_id_for(m, REGION)) is not None and r != m
]
# Cross-check the two helpers against each other. A row that prints a runtime id
# next to "mantle" only is self-contradictory, and it happened: endpoints_for()
# compared against a version-stripped catalogue key while runtime_id_for() used the
# full id, so gpt-oss showed a runtime id and "mantle". Neither helper complained.
contradictions = [
    m for m in COVERED
    if (runtime_id_for(m, REGION) is not None)
    != endpoints_for(m, REGION)["runtime"]
]
print()
if contradictions:
    print(f"!! runtime_id_for() and endpoints_for() DISAGREE for {contradictions}.")
    print("   One of them is wrong; do not trust the table above until they agree.")
print(f"=> {len(COVERED) - len(mantle_only)}/{len(COVERED)} of these are on "
      f"bedrock-runtime; {len(renamed)} under a different id.")
if mantle_only:
    print(f"   bedrock-mantle only: {mantle_only}")
    print("   For those, this notebook's endpoint is the only one that serves them.")
else:
    print("   Every model here is on both endpoints. This notebook shows the")
    print("   bedrock-mantle calls; the ids above are what you send to switch.")
print("   Region matters too: a model absent here can be present elsewhere, so")
print("   re-run this in the Region you intend to deploy in.")

model (as named on mantle)             on runtime as                            endpoints
------------------------------------------------------------------------------------------------


mistral.magistral-small-2509           mistral.magistral-small-2509             mantle, runtime


mistral.ministral-3-14b-instruct       mistral.ministral-3-14b-instruct         mantle, runtime


mistral.ministral-3-3b-instruct        mistral.ministral-3-3b-instruct          mantle, runtime


mistral.ministral-3-8b-instruct        mistral.ministral-3-8b-instruct          mantle, runtime


mistral.mistral-large-3-675b-instruct  mistral.mistral-large-3-675b-instruct    mantle, runtime



=> 5/5 of these are on bedrock-runtime; 0 under a different id.
   Every model here is on both endpoints. This notebook shows the
   bedrock-mantle calls; the ids above are what you send to switch.
   Region matters too: a model absent here can be present elsewhere, so
   re-run this in the Region you intend to deploy in.


## 1. First call

Auth is a short-term Bedrock API key minted from ambient IAM credentials —
valid ≤12 h, **not refreshable**, Region-pinned. (`../00-foundations/01` shows the
self-refreshing provider and the SigV4 alternative that needs no key.)

In [3]:
from aws_bedrock_token_generator import provide_token
from openai import OpenAI

# Build from a FRESH token; don't construct once at import and reuse for hours.
client = OpenAI(api_key=provide_token(region=REGION), base_url=BASE_URL)

completion = client.chat.completions.create(
    model=LARGE3,
    messages=[
        {
            "role": "user",
            "content": (
                "Explain why model size is not the only quality "
                "factor, in two sentences."
            ),
        }
    ],
    max_tokens=250,
)
# `content` can be None when the model spends the whole budget reasoning: the call
# succeeds with finish_reason="length" and no text. Check before printing -- this is
# the single most common surprise on this endpoint.
choice = completion.choices[0]
answer = choice.message.content or ""
if answer:
    print(answer)
else:
    print(f"(no text: finish_reason={choice.finish_reason!r} — raise max_tokens)")
print("\nusage:", completion.usage.model_dump_json())

Model size affects computational efficiency and resource demands, but performance depends more on training data quality, architecture optimization, and task alignment. A smaller, well-designed model with high-quality training can outperform a larger, poorly optimized one.

usage: {"completion_tokens":46,"prompt_tokens":19,"total_tokens":65,"completion_tokens_details":null,"prompt_tokens_details":null}


## 2. Why Chat Completions and not Responses

In [4]:
for label, path, body in [
    (
        "Chat Completions",
        f"{PREFIX}/chat/completions",
        {
            "model": LARGE3,
            "messages": [{"role": "user", "content": "Reply OK"}],
            "max_tokens": 16,
        },
    ),
    (
        "Responses (/v1)",
        f"{PREFIX}/responses",
        {"model": LARGE3, "input": "Reply OK", "max_output_tokens": 16},
    ),
    (
        "Responses (/openai/v1)",
        "/openai/v1/responses",
        {"model": LARGE3, "input": "Reply OK", "max_output_tokens": 16},
    ),
]:
    # Short timeout, single attempt: a wrong path does not always fail fast.
    code, data = post(path, body, region=REGION, attempts=1, timeout=45)
    shown = code if code != -1 else "stalled"
    print(f"  {label:24} -> {shown} {'' if code == 200 else err(data)[:60]}")

  Chat Completions         -> 200 


  Responses (/v1)          -> 400 The model 'mistral.mistral-large-3-675b-instruct' does not s


  Responses (/openai/v1)   -> 400 The model 'mistral.mistral-large-3-675b-instruct' does not s


Consequences of being Chat-Completions-only:

- **You own the conversation history** — no `previous_response_id`.
- **Reasoning content is not returned.** `reasoning_effort` is accepted and the
  model does think, but the Chat Completions schema has nowhere to put the trace.
- Structured output uses `response_format`, not `text.format`.

## 3. Sampling parameters

This family accepts both `temperature` and `top_p`. That is *not* universal on
mantle: the GPT-5.5 and GPT-5.6 families accept `temperature` only at its default
`1.0` and refuse `top_p` outright, and newer Claude models reject both as
deprecated. So never share one sampling config across families — probe each one,
as the cell below does.

Which models refuse what has already changed twice during this collection's life:
Gemma 4 and Grok both tightened in August 2026 and have since been relaxed again.
Read the probe output, not this paragraph.

In [5]:
for label, extra in [
    ("temperature=0.7", {"temperature": 0.7}),
    ("temperature=0.0", {"temperature": 0.0}),
    ("top_p=0.95", {"top_p": 0.95}),
    ("both", {"temperature": 0.7, "top_p": 0.95}),
    ("max_tokens=1", {"max_tokens": 1}),
]:
    body = {
        "model": LARGE3,
        "messages": [{"role": "user", "content": "Reply OK"}],
        "max_tokens": 16,
    }
    body.update(extra)
    code, data = post(f"{PREFIX}/chat/completions", body, region=REGION)
    print(f"  {label:18} -> HTTP {code} {'' if code == 200 else err(data)[:60]}")

  temperature=0.7    -> HTTP 200 


  temperature=0.0    -> HTTP 200 


  top_p=0.95         -> HTTP 200 


  both               -> HTTP 200 


  max_tokens=1       -> HTTP 200 


`max_tokens=1` is valid here; the Responses API enforces a minimum of 16. One
more reason the two surfaces are not interchangeable.

## 4. Streaming

In [6]:
stream = client.chat.completions.create(
    model=M8B,
    messages=[
        {
            "role": "user",
            "content": ("List four uses for a small language model."),
        }
    ],
    max_tokens=300,
    stream=True,
)
chunks = 0
try:
    for chunk in stream:
        delta = chunk.choices[0].delta.content
        if delta:
            chunks += 1
            print(delta, end="", flush=True)
except Exception as exc:
    # A stream can fail AFTER delivering part of the answer: a mid-stream
    # 5xx is not rare, and it has happened while building these notebooks.
    # Report what arrived instead of losing it - production code has to
    # decide whether a partial answer is usable or the call must be retried.
    print(f"\n[stream interrupted after {chunks} deltas: {type(exc).__name__}]")
print(f"\n\n[{chunks} content deltas]")

Here are four practical uses for a

 **small language model** (e.g.,

 a lightweight LLMs like TinyLlama,

 Pythia,

 or a distilled version of a larger model like

...

 team ideating for brainstorm meetings.
  

 - *Advantages*: Low computational overhead

 compared to larger models while being surprisingly helpful.



### 4.



[39 content deltas]


## 5. Multi-turn — you manage the history

In [7]:
messages = [
    {"role": "system", "content": "You are concise. Two sentences maximum."},
    {"role": "user", "content": "What is quantisation?"},
]
first = client.chat.completions.create(model=M14B, messages=messages, max_tokens=200)
print("assistant:", first.choices[0].message.content)

messages.append({"role": "assistant", "content": first.choices[0].message.content})
messages.append({"role": "user", "content": "What quality cost does it usually carry?"})

second = client.chat.completions.create(model=M14B, messages=messages, max_tokens=200)
print("\nassistant:", second.choices[0].message.content)
print(
    f"\ninput tokens grew: {first.usage.prompt_tokens} -> {second.usage.prompt_tokens}"
)

assistant: Quantization is the process of converting **continuous data** (e.g., analog signals) into a **discrete format**, like digital numbers (ex: 0s and 1s). It’s used in physics, signal processing, and machine learning (e.g., model weights in AI).



assistant: Quantization reduces **precision**, introducing errors like rounding mistakes or signal loss—degrading accuracy in data analysis or reconstruction (e.g., audio/video quality). Trade-off: smaller storage but potential fidelity loss.

input tokens grew: 18 -> 89


## 6. Reasoning effort, and the Magistral variant

`reasoning_effort` is accepted family-wide. `magistral-small-2509` is the
reasoning-oriented member — worth comparing against a similarly sized Ministral.

In [8]:
print(f"{'model':40} {'effort':8} {'completion tokens':>18}")
print("-" * 70)
PUZZLE = (
    "A rope burns in 60 minutes unevenly. How do you time 30 minutes with two ropes?"
)
for model in (M8B, MAGISTRAL):
    for effort in ("low", "high"):
        code, data = post(
            f"{PREFIX}/chat/completions",
            {
                "model": model,
                "messages": [{"role": "user", "content": PUZZLE}],
                "max_tokens": 700,
                "reasoning_effort": effort,
            },
            region=REGION,
        )
        tokens = (data.get("usage") or {}).get("completion_tokens", "-")
        print(f"{model:40} {effort:8} {tokens!s:>18}")

model                                    effort    completion tokens
----------------------------------------------------------------------


mistral.ministral-3-8b-instruct          low                     700


mistral.ministral-3-8b-instruct          high                    700


mistral.magistral-small-2509             low                     700


mistral.magistral-small-2509             high                    700


In [9]:
# What Magistral actually says on a reasoning task.
code, data = post(
    f"{PREFIX}/chat/completions",
    {
        "model": MAGISTRAL,
        "messages": [{"role": "user", "content": PUZZLE}],
        "max_tokens": 800,
        "reasoning_effort": "high",
    },
    region=REGION,
)
choice = (data.get("choices") or [{}])[0]
print("finish_reason:", choice.get("finish_reason"))
print((choice.get("message", {}).get("content") or "")[:500])

finish_reason: length
Alright, let's tackle this puzzle step by step. We have two ropes that each burn in 60 minutes, but they burn unevenly, meaning different sections may burn faster or slower than others. We need to measure exactly 30 minutes using these two ropes.

### Understanding the Problem

First, let's clarify the given information:
- **Rope A**: Burns in 60 minutes (unevenly)
- **Rope B**: Burns in 60 minutes (unevenly)
- **Goal**: Measure exactly 30 minutes using these two ropes.

The key points are:
1. B


## 7. Tool use (function calling)

Chat Completions nests the schema under `"function"` — unlike the Responses API,
which puts `name`/`parameters` at the top level.

In [10]:
def convert_currency(amount: float, source: str, target: str) -> dict:
    """Stand-in for an FX service."""
    rates = {("EUR", "USD"): 1.09, ("USD", "EUR"): 0.92, ("EUR", "SGD"): 1.46}
    rate = rates.get((source.upper(), target.upper()))
    return {
        "amount": amount,
        "from": source.upper(),
        "to": target.upper(),
        "rate": rate,
        "converted": round(amount * rate, 2) if rate else None,
    }


tools = [
    {
        "type": "function",
        "function": {
            "name": "convert_currency",
            "description": "Convert an amount between two currencies.",
            "parameters": {
                "type": "object",
                "properties": {
                    "amount": {"type": "number"},
                    "source": {"type": "string", "description": "ISO code, e.g. EUR"},
                    "target": {"type": "string", "description": "ISO code, e.g. USD"},
                },
                "required": ["amount", "source", "target"],
            },
        },
    }
]

convo = [{"role": "user", "content": "How much is 250 EUR in USD?"}]

# `tool_choice="auto"` means the model MAY call a tool, not that it will. Retry
# rather than assuming the first response carries a call.
msg = None
for attempt in range(1, 4):
    first = client.chat.completions.create(
        model=LARGE3, messages=convo, tools=tools, tool_choice="auto", max_tokens=800
    )
    choice = first.choices[0]
    print(
        f"attempt {attempt}: finish_reason={choice.finish_reason!r} "
        f"tool_calls={len(choice.message.tool_calls or [])}"
    )
    if choice.message.tool_calls:
        msg = choice.message
        break
if msg is None:
    raise RuntimeError("no tool call after 3 attempts - raise max_tokens")
print(
    "tool_calls:",
    [(c.function.name, c.function.arguments) for c in (msg.tool_calls or [])],
)

if msg.tool_calls:
    convo.append(msg.model_dump(exclude_none=True))
    for call in msg.tool_calls:
        args = parse_json_lenient(call.function.arguments)
        convo.append(
            {
                "role": "tool",
                "tool_call_id": call.id,
                "content": json.dumps(convert_currency(**args)),
            }
        )
    final = client.chat.completions.create(
        model=LARGE3, messages=convo, tools=tools, max_tokens=200
    )
    print("\nfinal answer:", final.choices[0].message.content)

attempt 1: finish_reason='tool_calls' tool_calls=1
tool_calls: [('convert_currency', '{"amount": 250, "source": "EUR", "target": "USD"}')]



final answer: 250 EUR is approximately **272.50 USD**.


### Small models are where tool use starts to fail

This is the practical reason to care about the size ladder. Ask every model to
make the same tool call and count who gets it right.

In [11]:
print(f"{'model':40} {'tool call?':>11}  arguments")
print("-" * 92)
for model in LADDER:
    code, data = post(
        f"{PREFIX}/chat/completions",
        {
            "model": model,
            "messages": [{"role": "user", "content": "How much is 250 EUR in USD?"}],
            "tools": tools,
            "tool_choice": "auto",
            "max_tokens": 300,
        },
        region=REGION,
    )
    if code != 200:
        print(f"{model:40} {'HTTP ' + str(code):>11}")
        continue
    choice = (data.get("choices") or [{}])[0]
    calls = choice.get("message", {}).get("tool_calls") or []
    if calls:
        print(f"{model:40} {'yes':>11}  {calls[0]['function']['arguments'][:44]}")
    else:
        text = (choice.get("message", {}).get("content") or "").strip()
        print(f"{model:40} {'no':>11}  prose: {text[:40]!r}")

model                                     tool call?  arguments
--------------------------------------------------------------------------------------------


mistral.ministral-3-3b-instruct                  yes  {"amount": 250, "source": "EUR", "target": "


mistral.ministral-3-8b-instruct                  yes  {"amount": 250, "source": "EUR", "target": "


mistral.ministral-3-14b-instruct                 yes  {"amount": 250, "source": "EUR", "target": "


mistral.mistral-large-3-675b-instruct            yes  {"amount": 250, "source": "EUR", "target": "


## 8. Structured output with `response_format`

In [12]:
def json_object_call(prompt, max_tokens, model=LARGE3):
    code, data = post(
        f"{PREFIX}/chat/completions",
        {
            "model": model,
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": max_tokens,
            "response_format": {"type": "json_object"},
        },
        region=REGION,
    )
    choice = (data.get("choices") or [{}])[0]
    content = choice.get("message", {}).get("content") or ""
    return code, choice.get("finish_reason"), content


for budget in (64, 600):
    code, finish, content = json_object_call(
        "Give the capital and population of France as JSON.", budget
    )
    print(f"max_tokens={budget:4} HTTP {code} finish={finish!s:8} len={len(content)}")
    if finish == "length" and not content.strip():
        print("    -> truncated before any JSON; raise max_tokens")
    elif content.strip():
        print("    ->", parse_json_lenient(content))

max_tokens=  64 HTTP 200 finish=stop     len=65
    -> {'country': 'France', 'capital': 'Paris', 'population': 68042591}


max_tokens= 600 HTTP 200 finish=stop     len=79
    -> {'country': 'France', 'capital': 'Paris', 'population': 68070697}


**Always check `finish_reason` before parsing.** A reasoning-capable model can
spend its whole budget thinking and return HTTP 200 with empty content.

In [13]:
schema = {
    "type": "object",
    "properties": {
        "country": {"type": "string"},
        "capital": {"type": "string"},
        "population_millions": {"type": "number"},
    },
    "required": ["country", "capital", "population_millions"],
    "additionalProperties": False,
}


def schema_call(budget: int):
    """One json_schema request at a given budget."""
    code, data = post(
        f"{PREFIX}/chat/completions",
        {
            "model": LARGE3,
            "messages": [{"role": "user", "content": "Describe France."}],
            "max_tokens": budget,
            "response_format": {
                "type": "json_schema",
                "json_schema": {"name": "country", "strict": True, "schema": schema},
            },
        },
        region=REGION,
    )
    choice = (data.get("choices") or [{}])[0]
    return code, choice, (choice.get("message", {}) or {}).get("content")  # may be None


# Escalate rather than retrying once. On a reasoning-heavy model the trace can eat
# a raised budget too, and one retry is not enough to tell "needs more room" from
# "will never comply".
content, choice = None, {}
for budget in (250, 1000, 4000):
    code, choice, content = schema_call(budget)
    print(f"max_tokens={budget:5} HTTP {code} finish={choice.get('finish_reason')!s:8} "
          f"chars={len(content or '')}")
    if (content or "").strip():
        break

parsed = parse_json_lenient(content or "") if (content or "").strip() else {}
print("\nparsed:", json.dumps(parsed, indent=2))

if not parsed:
    # Empty is a BUDGET problem, not a schema violation. A sample that raises here
    # teaches nothing; the lesson is to check finish_reason and escalate, or to turn
    # reasoning off for extraction work where you do not need it.
    print(f"\nNo JSON at any budget (finish_reason={choice.get('finish_reason')!r}).")
    print("The reasoning trace consumed the whole budget before the opening brace.")
    print("Options: raise max_tokens further, or send reasoning_effort='none' --")
    print("extraction rarely needs the thinking, and it frees the budget for output.")
else:
    missing = {"country", "capital"} - set(parsed)
    if missing:
        # A NON-empty object missing required keys IS a strict-mode violation and
        # worth failing on: the schema promised those keys.
        raise ValueError(f"model omitted required keys: {sorted(missing)} in {parsed}")
    print("required keys present: country, capital")

max_tokens=  250 HTTP 200 finish=stop     chars=80

parsed: {
  "country": "France",
  "capital": "Paris",
  "population_millions": 68.4
}
required keys present: country, capital


Parse **leniently**: even in strict mode some mantle models append characters
after a valid object, which makes a bare `json.loads()` raise on usable output.

## 9. Cost-aware routing across the size ladder

The pattern that justifies a family this wide: try the cheapest model, validate
the result against a schema, escalate only on failure.

In [14]:
EXTRACTION_SCHEMA = {
    "type": "object",
    "properties": {
        "invoice_number": {"type": "string"},
        "total_amount": {"type": "number"},
        "currency": {"type": "string"},
        "due_date": {"type": "string"},
    },
    "required": ["invoice_number", "total_amount", "currency", "due_date"],
    "additionalProperties": False,
}

INVOICE = (
    "INVOICE INV-2026-0042\nIssued: 2026-07-01\nDue: 2026-07-31\n"
    "Subtotal: 1,200.00 EUR\nVAT (19%): 228.00 EUR\nTotal: 1,428.00 EUR"
)


def try_extract(model: str):
    code, data = post(
        f"{PREFIX}/chat/completions",
        {
            "model": model,
            "messages": [
                {"role": "user", "content": f"Extract the invoice fields.\n\n{INVOICE}"}
            ],
            "max_tokens": 700,
            "response_format": {
                "type": "json_schema",
                "json_schema": {
                    "name": "invoice",
                    "strict": True,
                    "schema": EXTRACTION_SCHEMA,
                },
            },
        },
        region=REGION,
    )
    if code != 200:
        return None, f"HTTP {code}"
    choice = (data.get("choices") or [{}])[0]
    content = choice.get("message", {}).get("content") or ""
    if choice.get("finish_reason") == "length" and not content.strip():
        return None, "truncated"
    try:
        parsed = parse_json_lenient(content)
    except ValueError as exc:
        return None, f"unparseable ({exc.__class__.__name__})"
    # Validation gate: right keys AND the total we expect.
    if set(parsed) != set(EXTRACTION_SCHEMA["properties"]):
        return None, "wrong keys"
    if abs(float(parsed["total_amount"]) - 1428.0) > 0.01:
        return None, f"wrong total: {parsed['total_amount']}"
    return parsed, "ok"


print(f"{'model':40} {'verdict':>16}  result")
print("-" * 100)
chosen = None
for model in LADDER:  # cheapest first
    result, verdict = try_extract(model)
    print(f"{model:40} {verdict:>16}  {json.dumps(result)[:44] if result else ''}")
    if result and chosen is None:
        chosen = (model, result)

print(f"\ncheapest model that passed the gate: {chosen[0] if chosen else 'none'}")
if chosen:
    print("extracted:", json.dumps(chosen[1], indent=2))

model                                             verdict  result
----------------------------------------------------------------------------------------------------


mistral.ministral-3-3b-instruct                        ok  {"invoice_number": "INV-2026-0042", "total_a


mistral.ministral-3-8b-instruct                        ok  {"invoice_number": "INV-2026-0042", "total_a


mistral.ministral-3-14b-instruct                       ok  {"invoice_number": "INV-2026-0042", "total_a


mistral.mistral-large-3-675b-instruct                  ok  {"invoice_number": "INV-2026-0042", "total_a

cheapest model that passed the gate: mistral.ministral-3-3b-instruct
extracted: {
  "invoice_number": "INV-2026-0042",
  "total_amount": 1428.0,
  "currency": "EUR",
  "due_date": "2026-07-31-lgmt-timezone-not-represented-here.weekday-may-correctly-go-along-with-date-for-context kreeg control flowchart om zonename gebruik deude\u3042\u3093\u306e\u6210\u679c\u3092\u7d44\u307f\u8fbc\u3080Voor uitwerkingsreeksen, aanvullende details voor *Omschrijving "
}


## 10. Latency across the ladder

TTFT is dominated by *prefill* (the model reading your prompt) plus queue time.
Service tiers trade cost against queue priority.

Read the table below as **one sample, not a benchmark** — but do not expect the
three rows to look identical either. `flex` is deliberately deprioritised, so its
TTFT is usually the worst of the three by a clear margin; `default` and `priority`
sit close together on an idle account and separate under contention. A single
outlier in either direction is normal, and one `priority` call elsewhere in this
collection took 54 seconds. Take medians over many calls before quoting a number.
(`../00-foundations/03` has the full treatment.)

In [15]:
task = "In one sentence, what is a mixture-of-experts model?"
print(f"{'model':40} {'latency':>9} {'out tok':>8}  answer")
print("-" * 104)
for model in LADDER + [MAGISTRAL]:
    started = time.perf_counter()
    code, data = post(
        f"{PREFIX}/chat/completions",
        {
            "model": model,
            "messages": [{"role": "user", "content": task}],
            "max_tokens": 160,
        },
        region=REGION,
    )
    elapsed = time.perf_counter() - started
    if code != 200:
        print(f"{model:40} {'-':>9} {'-':>8}  HTTP {code}: {err(data)[:34]}")
        continue
    text = " ".join((data["choices"][0]["message"]["content"] or "").split())
    print(
        f"{model:40} {elapsed:>8.2f}s {data['usage']['completion_tokens']:>8}  "
        f"{text[:38]!r}"
    )

model                                      latency  out tok  answer
--------------------------------------------------------------------------------------------------------


mistral.ministral-3-3b-instruct              1.27s       54  'A **mixture-of-experts (MoE)** model i'


mistral.ministral-3-8b-instruct              1.09s       45  'A **mixture-of-experts (MoE) model** i'


mistral.ministral-3-14b-instruct             1.20s       54  'A **mixture-of-experts (MoE) model** i'


mistral.mistral-large-3-675b-instruct        1.26s       46  'A **mixture-of-experts (MoE) model** i'


mistral.magistral-small-2509                 1.64s       45  'A mixture-of-experts model is a machin'


In [16]:
print(f"{'tier':10} {'TTFT (s)':>10} {'total (s)':>10} {'frames/s':>10}")
print("-" * 44)
for tier in ("default", "flex", "priority"):
    m = ttft(
        f"{PREFIX}/chat/completions",
        {
            "model": M8B,
            "messages": [{"role": "user", "content": task}],
            "max_tokens": 200,
            "service_tier": tier,
        },
        region=REGION,
    )
    if m.get("error"):
        # Do NOT label every failure "tier not supported". A URLError or a timeout
        # is a transport problem and says nothing about whether the parameter is
        # accepted; reporting it as an unsupported feature invents a limitation
        # the service never claimed. Read the error before attributing a cause.
        detail = str(m["error"])
        if "service_tier" in detail or "unsupported" in detail.lower():
            cause = "tier refused by this model"
        else:
            cause = "transport error - retry; tells you nothing about tier support"
        print(f"{tier:10} {detail[:30]:>32}  ({cause})")
    else:
        print(
            f"{tier:10} {m['ttft_s']:>10.3f} {m['total_s']:>10.3f} "
            f"{m['frames_per_s']:>10.1f}"
        )

tier         TTFT (s)  total (s)   frames/s
--------------------------------------------


default         1.107      1.550       11.3


flex            2.311      2.314      661.0


priority        1.076      1.076     5671.7


## 11. Regional availability

Note `mistral-large-3` is **absent from eu-central-1** while the Ministrals are
present — so a EU-resident deployment cannot use the flagship as a fallback.

In [17]:
regions = ("us-east-1", "us-east-2", "us-west-2", "eu-central-1")
inventory = {}
for region in regions:
    try:
        inventory[region] = set(list_models(region))
    except (RuntimeError, OSError) as exc:
        inventory[region] = set()
        print(f"{region}: {type(exc).__name__}")

print(f"{'model':40} " + "  ".join(f"{r:>13}" for r in regions))
print("-" * 100)
for model in LADDER + [MAGISTRAL]:
    cells = "  ".join(
        f"{('yes' if model in inventory[r] else '-'):>13}" for r in regions
    )
    print(f"{model:40} {cells}")

model                                        us-east-1      us-east-2      us-west-2   eu-central-1
----------------------------------------------------------------------------------------------------
mistral.ministral-3-3b-instruct                    yes            yes            yes            yes
mistral.ministral-3-8b-instruct                    yes            yes            yes            yes
mistral.ministral-3-14b-instruct                   yes            yes            yes            yes
mistral.mistral-large-3-675b-instruct              yes            yes            yes              -
mistral.magistral-small-2509                       yes            yes            yes            yes


## 12. Production hardening

In [18]:
code, project = post(
    "/v1/organization/projects",
    {
        "name": "mistral-samples",
        "tags": {"Application": "MistralDemo", "Environment": "Demo"},
    },
    region=REGION,
)
project_id = project.get("id")
safe_print("project:", code, project_id)


class MistralRouter:
    """Cheapest-model-first router with a schema validation gate."""

    def __init__(self, ladder=LADDER, region=REGION, tier="default", project=None):
        self.ladder, self.region, self.tier, self.project = (
            ladder,
            region,
            tier,
            project,
        )

    def extract(self, prompt: str, schema: dict, validate=None, max_tokens=700):
        headers = {"OpenAI-Project": self.project} if self.project else None
        for model in self.ladder:
            # post() retries 429/5xx with exponential backoff and jitter.
            code, data = post(
                f"{PREFIX}/chat/completions",
                {
                    "model": model,
                    "messages": [{"role": "user", "content": prompt}],
                    "max_tokens": max_tokens,
                    "temperature": 0.2,
                    "service_tier": self.tier,
                    "response_format": {
                        "type": "json_schema",
                        "json_schema": {
                            "name": "out",
                            "strict": True,
                            "schema": schema,
                        },
                    },
                },
                region=self.region,
                headers=headers,
            )
            if code != 200:
                continue
            choice = (data.get("choices") or [{}])[0]
            content = choice.get("message", {}).get("content") or ""
            if not content.strip():  # e.g. finish_reason == "length"
                continue
            try:
                parsed = parse_json_lenient(content)
            except ValueError:
                continue
            if validate and not validate(parsed):
                continue
            return {"model": model, "result": parsed}
        raise RuntimeError("no model in the ladder produced a valid result")


router = MistralRouter(project=project_id)
out = router.extract(
    f"Extract the invoice fields.\n\n{INVOICE}",
    EXTRACTION_SCHEMA,
    validate=lambda d: abs(float(d.get("total_amount", 0)) - 1428.0) < 0.01,
)
print("served by:", out["model"])
print("result   :", json.dumps(out["result"]))

project: 200 proj_mazcrpa3...


served by: mistral.ministral-3-3b-instruct
result   : {"invoice_number": "INV-2026-0042", "total_amount": 1428.0, "currency": "EUR", "due_date": "2026-07-31"}


In [19]:
code, archived = post(
    f"/v1/organization/projects/{project_id}/archive", {}, region=REGION
)
print("archived:", code, archived.get("status"))

archived: 200 archived


## Gotchas — Mistral on bedrock-mantle

| Gotcha | Detail |
|---|---|
| Path prefix | Bare `/v1`, **not** `/openai/v1` |
| Responses API | Returns 400 for this family — Chat Completions only |
| Wrong path | Not always a fast failure — always set a client-side timeout |
| History | No `previous_response_id`; send `messages` every turn |
| Reasoning trace | Not returned for this family — `message.reasoning` is absent. Other `/v1` families (qwen, deepseek, glm, kimi, minimax, nemotron-super) **do** return it |
| `content` can be `None` | Check `finish_reason` before slicing/parsing |
| Strict JSON | Parse leniently — models can append text after a valid object |
| Small models | 3B/8B often skip tool calls — validate, then escalate |
| Region | `mistral-large-3` absent from eu-central-1; Ministrals present |
| Quotas | No RPM quota; most models have no published TPM (tokens per minute) — retry with backoff |

## Where next
- `02-devstral-and-voxtral.ipynb` — coding and audio-capable variants
- Same API shape: `../04-qwen/`, `../05-deepseek/`, `../06-zai-glm/`
- Different API shape: `../03-google-gemma/` (Responses), `../02-anthropic-claude/`
  (Messages)

## Also on `bedrock-runtime`? Mistral

The whole current Mistral line is on both endpoints under identical IDs, which makes it a clean family for comparing the two APIs side by side.

`endpoints_for()` asks both catalogues rather than trusting a table, so the cell
below tells you today's answer. Converse is worth reaching for when you want one
request shape across providers, or a feature that only `bedrock-runtime` carries.


In [20]:
from bedrock import (
    converse,
    converse_reasoning,
    endpoints_for,
    resolve_runtime_id,
)

MANTLE_ID = "mistral.mistral-large-3-675b-instruct"
RUNTIME_ID = "mistral.mistral-large-3-675b-instruct"

print("endpoint availability:", endpoints_for(MANTLE_ID))
print("mantle model id :", MANTLE_ID)
print("runtime model id:", RUNTIME_ID)
resolved = resolve_runtime_id(RUNTIME_ID)
print("converse sends  :", resolved)
if resolved != RUNTIME_ID:
    print("                  ^ resolved for you; the form above would be rejected")

# The same question, through Converse. Note the shape: content is a LIST of
# blocks rather than a string, and the token budget lives in inferenceConfig.
# The budget is generous on purpose - a reasoning model spends it on the trace
# first and returns no text block at all if it runs out.
text, response = converse(
    RUNTIME_ID,
    [
        {
            "role": "user",
            "content": [
                {"text": "Name one benefit of idempotency. One sentence."}
            ],
        }
    ],
    max_tokens=400,
    system="You are terse.",
)

error = (response.get("error") or {}).get("message")
if error:
    print("\ncall failed:", error[:200])
else:
    reasoning = converse_reasoning(response)
    print("\nstop reason:", response.get("stopReason"))
    print("tokens     :", response.get("usage", {}).get("totalTokens"))
    if reasoning:
        print(f"reasoning  : {len(reasoning)} chars (returned in a "
              "reasoningContent block, before the text)")
    if text.strip():
        print("answer     :", text.strip()[:200])
    else:
        # Empty text is NOT the same as a failed call. Say which it is.
        print("answer     : (none - the budget went to reasoning; raise max_tokens)")


endpoint availability: {'mantle': True, 'runtime': True}
mantle model id : mistral.mistral-large-3-675b-instruct
runtime model id: mistral.mistral-large-3-675b-instruct


converse sends  : mistral.mistral-large-3-675b-instruct



stop reason: end_turn
tokens     : 40
answer     : Idempotency ensures repeated operations produce the same result, preventing unintended side effects.


## Converse in earnest — the tool loop, provider parameters, and caching

The earlier endpoint section proved this model answers through Converse. That is
the easy part. This section does the three things you actually need on
`bedrock-runtime`, because each differs from the `bedrock-mantle` equivalent:

1. **A complete tool round trip** — `toolUse` out, `toolResult` back in. Getting a
   tool *call* is half the job; feeding the result back is where the shapes bite.
2. **`additionalModelRequestFields`** — Converse normalises the common fields, so
   anything provider-specific goes through this escape hatch.
3. **`cachePoint`** — prompt caching is a first-class Converse block, and support
   for it is per model rather than universal.


In [21]:
from bedrock import converse_text, converse_tool_uses, resolve_runtime_id, runtime_client

RUNTIME_ID = "mistral.mistral-large-3-675b-instruct"
runtime = runtime_client(REGION)
resolved = resolve_runtime_id(RUNTIME_ID, REGION)

# Converse tool shape: toolSpec, and the JSON Schema nests under inputSchema.json.
# This is NOT the OpenAI shape - there is no {"type": "function"} wrapper.
WEATHER_TOOL = {
    "toolSpec": {
        "name": "get_weather",
        "description": "Current weather for a city",
        "inputSchema": {
            "json": {
                "type": "object",
                "properties": {"city": {"type": "string"}},
                "required": ["city"],
            }
        },
    }
}

history = [
    {"role": "user", "content": [{"text": "What is the weather in Singapore? Use the tool."}]}
]
first = runtime.converse(
    modelId=resolved,
    messages=history,
    toolConfig={"tools": [WEATHER_TOOL]},
    inferenceConfig={"maxTokens": 500},
)
print("turn 1 stop reason:", first.get("stopReason"))
print("turn 1 blocks     :", [next(iter(b)) for b in first["output"]["message"]["content"]])

uses = converse_tool_uses(first)
if not uses:
    print("no tool call this run - tool_choice defaults to the model's discretion;")
    print("retry, or set toolConfig['toolChoice'] to compel one.")
else:
    use = uses[0]
    print(f"tool call         : {use['name']}({use['input']})")
    # Validate before acting on it. A malformed call still reports tool_use.
    city = str(use["input"].get("city", "")).lower()
    print("arguments valid   :", "yes" if "singapore" in city else f"NO ({use['input']})")

    # Echo the assistant turn back VERBATIM, then answer with a toolResult whose
    # toolUseId matches. Dropping either breaks the loop with a 400.
    history.append(first["output"]["message"])
    history.append(
        {
            "role": "user",
            "content": [
                {
                    "toolResult": {
                        "toolUseId": use["toolUseId"],
                        "content": [{"json": {"tempC": 31, "conditions": "humid"}}],
                    }
                }
            ],
        }
    )
    second = runtime.converse(
        modelId=resolved,
        messages=history,
        toolConfig={"tools": [WEATHER_TOOL]},
        inferenceConfig={"maxTokens": 300},
    )
    print("turn 2 stop reason:", second.get("stopReason"))
    print("final answer      :", converse_text(second).strip()[:160])


turn 1 stop reason: tool_use
turn 1 blocks     : ['toolUse']
tool call         : get_weather({'city': 'Singapore'})
arguments valid   : yes


turn 2 stop reason: end_turn
final answer      : The current weather in **Singapore** is:
- **Temperature**: 31°C
- **Conditions**: Humid

Would you like any additional details?


In [22]:
# Converse normalises maxTokens, temperature, topP and stopSequences. Anything
# provider-specific goes through additionalModelRequestFields, unvalidated by
# Converse and passed to the provider as-is. That makes it powerful and sharp:
# a key this model does not recognise is a 400, not a silent no-op.
from bedrock import converse_reasoning

PUZZLE = (
    "A bat and ball cost $1.10 together. The bat costs $1.00 more than the ball. "
    "How much is the ball?"
)

for label, extra in [
    ("no extra fields", None),
    ("provider fields", {"reasoning_effort": "low"}),
]:
    kwargs = {"additionalModelRequestFields": extra} if extra else {}
    try:
        response = runtime.converse(
            modelId=resolved,
            messages=[{"role": "user", "content": [{"text": PUZZLE}]}],
            inferenceConfig={"maxTokens": 900},
            **kwargs,
        )
    except Exception as exc:
        print(f"{label:<16} {type(exc).__name__}: {str(exc)[-90:]}")
        continue
    blocks = [next(iter(b)) for b in response["output"]["message"]["content"]]
    trace = converse_reasoning(response)
    answer = converse_text(response).strip().replace("\n", " ")
    print(f"{label:<16} out={response['usage']['outputTokens']:>4} blocks={blocks}")
    print(f"{'':<16} reasoning={len(trace)} chars | {answer[:70]}")

print()
print("Note whether a reasoningContent block appears above. Some models return the")
print("trace as a typed block on Converse and some do not, so read the blocks")
print("rather than assuming - and never index content[0].")


no extra fields  out= 705 blocks=['text']
                 reasoning=0 chars | Alright, let's tackle this problem step by step. The problem states:  


provider fields  out= 688 blocks=['text']
                 reasoning=0 chars | Alright, let's tackle this problem step by step. The problem states:  

Note whether a reasoningContent block appears above. Some models return the
trace as a typed block on Converse and some do not, so read the blocks
rather than assuming - and never index content[0].


In [23]:
# cachePoint is a Converse block, but support for it is per model rather than
# universal. Ask before designing around it.
HANDBOOK = "You are a support handbook. " + (
    "Retries: use exponential backoff with full jitter, cap at 16 seconds. " * 160
)

try:
    usage = runtime.converse(
        modelId=resolved,
        system=[{"text": HANDBOOK}, {"cachePoint": {"type": "default"}}],
        messages=[{"role": "user", "content": [{"text": "One line: the retry policy?"}]}],
        inferenceConfig={"maxTokens": 60},
    )["usage"]
    print("cachePoint accepted:", {k: v for k, v in usage.items() if "cache" in k.lower()})
except Exception as exc:
    print(f"cachePoint -> {type(exc).__name__}")
    print(f"    {str(exc)[-140:]}")
    print()
    print("Not every model supports prompt caching on Converse. Note the exception")
    print("type: this surfaces as an access or validation error rather than a clear")
    print("'unsupported feature' message, which is easy to misread as a permissions")
    print("problem. Probe it once per model instead of assuming it is available.")

# The same call without the cachePoint block works, so caching is the only part
# that is unavailable.
usage = runtime.converse(
    modelId=resolved,
    system=[{"text": "You are terse."}],
    messages=[{"role": "user", "content": [{"text": "One line: why use backoff?"}]}],
    inferenceConfig={"maxTokens": 60},
)["usage"]
print()
print("same call without cachePoint -> OK, tokens:", usage["totalTokens"])


cachePoint -> AccessDeniedException
    nverse operation: You invoked an unsupported model or your request did not allow prompt caching. See the documentation for more information.

Not every model supports prompt caching on Converse. Note the exception
type: this surfaces as an access or validation error rather than a clear
'unsupported feature' message, which is easy to misread as a permissions
problem. Probe it once per model instead of assuming it is available.



same call without cachePoint -> OK, tokens: 31


### What this section adds over the endpoint check above

- **The tool loop is the part that bites.** `toolSpec` is not the OpenAI shape,
  the JSON Schema nests under `inputSchema.json`, and the second turn must echo the
  assistant message back verbatim alongside a `toolResult` whose `toolUseId`
  matches. Miss any of that and you get a 400.
- **`additionalModelRequestFields` is unvalidated by Converse.** It is the only way
  to reach provider-specific behaviour, and a key the model does not recognise
  fails the call rather than being ignored.
- **Feature support is per model, not per endpoint.** Read the output above rather
  than carrying an assumption over from another family.
